# Exploratory Data Analysis - Retail Demand Forecasting

This notebook performs exploratory data analysis on Walmart sales data to understand:
- Store performance patterns
- Holiday impact on sales
- Department-level insights
- Data quality and missing values

In [ ]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)
print("All imports successful")

In [ ]:
# Cell 2 — Load raw data
train    = pd.read_csv('../data/raw/train.csv',    parse_dates=['Date'])
features = pd.read_csv('../data/raw/features.csv', parse_dates=['Date'])
stores   = pd.read_csv('../data/raw/stores.csv')

print("Train shape   :", train.shape)
print("Features shape:", features.shape)
print("Stores shape  :", stores.shape)
train.head()

In [ ]:
# Cell 3 — Merge all three files
df = train.merge(features, on=['Store','Date','IsHoliday'], how='left')
df = df.merge(stores, on='Store', how='left')

print("Merged shape:", df.shape)
print("Columns     :", df.columns.tolist())
print("Date range  :", df['Date'].min(), "→", df['Date'].max())
df.head()

In [ ]:
# Cell 4 — Missing value audit
missing     = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

audit = pd.DataFrame({
    'Missing Count': missing,
    'Missing %'    : missing_pct
})

audit[audit['Missing Count'] > 0].sort_values('Missing %', ascending=False)

In [ ]:
# Cell 9 — SQL layer
conn = sqlite3.connect('../data/processed/walmart.db')
df.to_sql('sales', conn, if_exists='replace', index=False)
print("Loaded into SQLite — walmart.db\n")

q1 = pd.read_sql_query("""
    SELECT Type,
           ROUND(AVG(Weekly_Sales),2) AS Avg_Weekly_Sales,
           ROUND(SUM(Weekly_Sales),2) AS Total_Sales,
           COUNT(*)                   AS Records
    FROM sales
    GROUP BY Type
    ORDER BY Avg_Weekly_Sales DESC
""", conn)
print("Q1 — Avg sales by store type:")
print(q1)

q2 = pd.read_sql_query("""
    SELECT Type, IsHoliday,
           ROUND(AVG(Weekly_Sales),2) AS Avg_Sales
    FROM sales
    GROUP BY Type, IsHoliday
    ORDER BY Type, IsHoliday
""", conn)
print("\nQ2 — Holiday premium by store type:")
print(q2)

q3 = pd.read_sql_query("""
    SELECT Dept,
           ROUND(AVG(Weekly_Sales),2) AS Avg_Weekly_Sales
    FROM sales
    GROUP BY Dept
    ORDER BY Avg_Weekly_Sales DESC
    LIMIT 5
""", conn)
print("\nQ3 — Top 5 departments:")
print(q3)

q4 = pd.read_sql_query("""
    SELECT Store,
           ROUND(AVG(Weekly_Sales),2)                  AS Avg_Sales,
           ROUND(MAX(Weekly_Sales)-MIN(Weekly_Sales),2) AS Sales_Range,
           COUNT(*)                                     AS Weeks_Recorded
    FROM sales
    GROUP BY Store
    ORDER BY Sales_Range DESC
    LIMIT 10
""", conn)
print("\nQ4 — Highest variance stores (stockout risk candidates):")
print(q4)

conn.close()
print("\nSQLite connection closed.")

In [ ]:
# Cell 10 — Clean + export
for col in ['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']:
    df[col] = df[col].fillna(0)

df = df.sort_values(['Store','Dept','Date'])
df[['CPI','Unemployment']] = (
    df.groupby('Store')[['CPI','Unemployment']]
    .transform(lambda x: x.ffill().bfill())
)

df = df[df['Weekly_Sales'] >= 0].reset_index(drop=True)

df.to_csv('../data/processed/walmart_clean.csv', index=False)

print(f"Clean dataset saved")
print(f"Rows       : {df.shape[0]:,}")
print(f"Columns    : {df.shape[1]}")
print(f"Date range : {df['Date'].min()} → {df['Date'].max()}")
print(f"Stores     : {df['Store'].nunique()}")
print(f"Departments: {df['Dept'].nunique()}")
print(f"Null check : {df.isnull().sum().sum()} nulls remaining")

In [ ]:
# Cell 5 — Store type breakdown
store_summary = stores.groupby('Type').agg(
    Count   =('Store','count'),
    Avg_Size=('Size','mean')
).round(0)
print(store_summary)

stores['Type'].value_counts().plot(
    kind='bar',
    color=['#185FA5','#1D9E75','#BA7517'],
    figsize=(6,4),
    edgecolor='white'
)
plt.title('Store count by type')
plt.xlabel('Store type')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6 — Weekly sales distribution
fig, axes = plt.subplots(1, 2, figsize=(14,4))

axes[0].hist(df['Weekly_Sales'], bins=100,
             color='#185FA5', edgecolor='white')
axes[0].set_title('Weekly Sales Distribution')
axes[0].set_xlabel('Weekly Sales ($)')
axes[0].set_ylabel('Frequency')

axes[1].hist(
    df[df['Weekly_Sales'] > 0]['Weekly_Sales'].apply(np.log),
    bins=100, color='#1D9E75', edgecolor='white'
)
axes[1].set_title('Log Weekly Sales Distribution')
axes[1].set_xlabel('Log Weekly Sales')

plt.tight_layout()
plt.show()

print(f"Mean weekly sales   : ${df['Weekly_Sales'].mean():>12,.0f}")
print(f"Median weekly sales : ${df['Weekly_Sales'].median():>12,.0f}")
print(f"Total sales         : ${df['Weekly_Sales'].sum():>12,.0f}")

In [ ]:
# Cell 7 — Holiday impact analysis
holiday_stats = df.groupby('IsHoliday')['Weekly_Sales'].agg(['mean','median','count'])
holiday_stats.index = ['Non-Holiday','Holiday']
print(holiday_stats.round(2))

lift = (
    (holiday_stats.loc['Holiday','mean'] - holiday_stats.loc['Non-Holiday','mean'])
    / holiday_stats.loc['Non-Holiday','mean'] * 100
)
print(f"\nHoliday sales lift: +{lift:.1f}%")

df.groupby(['Date','IsHoliday'])['Weekly_Sales'].mean().unstack().plot(
    figsize=(14,4),
    color=['#185FA5','#D85A30'],
    linewidth=1.5
)
plt.title('Average weekly sales — holiday vs non-holiday weeks')
plt.ylabel('Average weekly sales ($)')
plt.legend(['Non-Holiday','Holiday'])
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8 — Top 10 stores by total sales
top_stores = (
    df.groupby('Store')['Weekly_Sales']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_stores.plot(kind='bar', color='#185FA5', figsize=(10,4), edgecolor='white')
plt.title('Top 10 stores by total sales')
plt.xlabel('Store')
plt.ylabel('Total sales ($)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("Top 3 stores account for ${:,.0f} in total sales".format(
    top_stores.head(3).sum()
))